# Lab 6: Non-Linear Models and Feature Engineering

We will be using the housing dataset from the previous labs for this example.

In [1]:
# import the libraries we need
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
# Load the House Prices dataset and view it
df = pd.read_csv('train.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
# Delete the outliers -> df['GrLivArea']>4000
df_sample = df.drop(df[df['GrLivArea']>4000].index)

In [4]:
# filter the dataset based on the following columns
columns_to_use = ['LotArea', 'YrSold', 'GarageArea', 'GarageYrBlt', 'GrLivArea', 'Neighborhood', 'MSZoning', 
                  'OverallQual', 'ExterQual', 'KitchenQual', 'MasVnrArea', 'YearBuilt', 'SalePrice']

# save the new dataset into df_sample
df_sample = df.loc[:, columns_to_use] 

In [5]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotArea       1460 non-null   int64  
 1   YrSold        1460 non-null   int64  
 2   GarageArea    1460 non-null   int64  
 3   GarageYrBlt   1379 non-null   float64
 4   GrLivArea     1460 non-null   int64  
 5   Neighborhood  1460 non-null   object 
 6   MSZoning      1460 non-null   object 
 7   OverallQual   1460 non-null   int64  
 8   ExterQual     1460 non-null   object 
 9   KitchenQual   1460 non-null   object 
 10  MasVnrArea    1452 non-null   float64
 11  YearBuilt     1460 non-null   int64  
 12  SalePrice     1460 non-null   int64  
dtypes: float64(2), int64(7), object(4)
memory usage: 148.4+ KB


Before we fill-in the missing values, we need to convert the non-numeric columns into numeric values. Are the non-numeric values ordinal or nominal?

In this case, `KitchenQual` and `ExterQual` are ordered from excellent quality to poor quality, so we can treat them as ordinal values. The other two (`Neighborhood` and `MSZoning`) are nominal.

In [6]:
# How many unique values do we have for KitchenQual?
df_sample['KitchenQual'].value_counts()

KitchenQual
TA    735
Gd    586
Ex    100
Fa     39
Name: count, dtype: int64

In [7]:
# How many unique values do we have for ExterQual?
df_sample['ExterQual'].value_counts()

ExterQual
TA    906
Gd    488
Ex     52
Fa     14
Name: count, dtype: int64

In [8]:
# create a second dataset from the first dataset
df_transformed = df_sample.copy()

In [9]:
# use ordinal encoder to transform kitchen quality (KitchenQual) and exterior material quality (ExterQual)
from sklearn.preprocessing import OrdinalEncoder
# Fair, Typical/Average, Good, Excellent
order = ['Fa','TA','Gd','Ex']
columns_with_order = ['ExterQual', 'KitchenQual']

for col in columns_with_order:
    ord_en = OrdinalEncoder(categories = [order]) # If we do not specify the order, it will use alphabetical order
    df_transformed[col] = ord_en.fit_transform(df_transformed[[col]])
    
# Specify the columns to be one-hot encoded
columns_to_encode = ['Neighborhood', 'MSZoning']

# Perform one-hot encoding
encoded_df = pd.get_dummies(df_transformed, columns=columns_to_encode)
encoded_df

,LotArea,YrSold,GarageArea,GarageYrBlt,GrLivArea,OverallQual,ExterQual,KitchenQual,MasVnrArea,YearBuilt,...,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker,MSZoning_C (all),MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM
0,8450,2008,548,2003.0,1710,7,2.0,2.0,196.0,2003,...,False,False,False,False,False,False,False,False,True,False
1,9600,2007,460,1976.0,1262,6,1.0,1.0,0.0,1976,...,False,False,False,False,True,False,False,False,True,False
2,11250,2008,608,2001.0,1786,7,2.0,2.0,162.0,2001,...,False,False,False,False,False,False,False,False,True,False
3,9550,2006,642,1998.0,1717,7,1.0,2.0,0.0,1915,...,False,False,False,False,False,False,False,False,True,False
4,14260,2008,836,2000.0,2198,8,2.0,2.0,350.0,2000,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,7917,2007,460,1999.0,1647,6,1.0,1.0,0.0,1999,...,False,False,False,False,False,False,False,False,True,False
1456,13175,2010,500,1978.0,2073,6,1.0,1.0,119.0,1978,...,False,False,False,False,False,False,False,False,True,False
1457,9042,2010,252,1941.0,2340,7,3.0,2.0,0.0,1941,...,False,False,False,False,False,False,False,False,True,False
1458,9717,2010,240,1950.0,1078,5,1.0,2.0,0.0,1950,...,False,False,False,False,False,False,False,False,True,False


Now we can fill-in any missing values:

In [10]:
# fillna with mean for: GarageYrBlt, MasVnrArea
encoded_df.fillna(encoded_df.mean(), inplace=True)

In [11]:
encoded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 41 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   LotArea               1460 non-null   int64  
 1   YrSold                1460 non-null   int64  
 2   GarageArea            1460 non-null   int64  
 3   GarageYrBlt           1460 non-null   float64
 4   GrLivArea             1460 non-null   int64  
 5   OverallQual           1460 non-null   int64  
 6   ExterQual             1460 non-null   float64
 7   KitchenQual           1460 non-null   float64
 8   MasVnrArea            1460 non-null   float64
 9   YearBuilt             1460 non-null   int64  
 10  SalePrice             1460 non-null   int64  
 11  Neighborhood_Blmngtn  1460 non-null   bool   
 12  Neighborhood_Blueste  1460 non-null   bool   
 13  Neighborhood_BrDale   1460 non-null   bool   
 14  Neighborhood_BrkSide  1460 non-null   bool   
 15  Neighborhood_ClearCr 

The next step is to create the feature matrix and target vector, then split the data into training and validation sets.

In [12]:
# create target vector and feature matrix
X = encoded_df.drop(['SalePrice'], axis=1)
y = encoded_df['SalePrice']

#Split the dataset into training and validation subsets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

Let's test the data with Ridge Regression and Random Forest models. Do either of these models require scaling?

In [13]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Initialize and train a Ridge regression model
ridge_model = Ridge(alpha=10)
ridge_model.fit(X_train, y_train)

# Initialize and train a Random Forest model
random_forest = RandomForestRegressor(n_estimators=100, random_state=42)
random_forest.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [14]:
# Evaluate the performance with unscaled data
RF_train_acc = random_forest.score(X_train, y_train)
RF_val_acc = random_forest.score(X_val, y_val)
Rid_train_acc = ridge_model.score(X_train, y_train)
Rid_val_acc = ridge_model.score(X_val, y_val)

# Print the results
print("Random Forest Training Accuracy:", RF_train_acc)
print("Random Forest Validation Accuracy:", RF_val_acc)
print("Ridge Training Accuracy:", Rid_train_acc)
print("Ridge Validation Accuracy:", Rid_val_acc)

Random Forest Training Accuracy: 0.9744302338246689
Random Forest Validation Accuracy: 0.807156069365349
Ridge Training Accuracy: 0.837937480985437
Ridge Validation Accuracy: 0.7324840354205198


In [15]:
from sklearn.preprocessing import StandardScaler
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

ridge_model.fit(X_train_scaled, y_train)
random_forest.fit(X_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [16]:
# Evaluate the performance with scaled data
RF_train_acc = random_forest.score(X_train_scaled, y_train)
RF_val_acc = random_forest.score(X_val_scaled, y_val)
Rid_train_acc = ridge_model.score(X_train_scaled, y_train)
Rid_val_acc = ridge_model.score(X_val_scaled, y_val)

# Print the results
print("Random Forest Training Accuracy:", RF_train_acc)
print("Random Forest Validation Accuracy:", RF_val_acc)
print("Ridge Training Accuracy:", Rid_train_acc)
print("Ridge Validation Accuracy:", Rid_val_acc)

Random Forest Training Accuracy: 0.9744624461590364
Random Forest Validation Accuracy: 0.8074661067604219
Ridge Training Accuracy: 0.8400764493693816
Ridge Validation Accuracy: 0.7406157066770049
